# &#x1F50D; Explore MGnify Biomes &#x1F50D;

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ebi-metagenomics/mgnipy/blob/main/docs/tutorial/demo-mgnify-biomes.ipynb)

In this demo we explore: What [biome classifications](https://bioportal.bioontology.org/ontologies/GOLDTERMS) can we find in MGnify? using mgnipy.

We will:
1. Initialize a biome query with either the Biomes proxy or the MGnipy client.
2. Filter and preview request URLs before fetching full results.
3. Run asynchronous requests to retrieve biome data efficiently.
4. Inspect results as a list and visualize them as a hierarchical biome tree.

In [1]:
# uncomment below if colab
#!pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple mgnipy
#!pip install asyncio

We can use the proxies.Biomes or MGnipy.

## Option 1. Proxies

The `Biomes` proxy provides a direct way to query biome information from the MGnify API. You can customize your query using various parameters such as `page_size` and `max_depth` to control the number of results and the depth of the biome hierarchy.

- Use `list_parameters()` to see all available filters and options.
- The `filter()` method allows you to refine your query further.
- The `explain()` method previews the constructed API URLs and the first few results.

This approach is useful if you want fine-grained control over the API request and wish to explore the available biome data interactively.

In [2]:
from mgnipy.V2.proxies import Biomes

biomes = Biomes(
    page_size=50,
)
print("Init url: ", biomes.request_url)
# if not sure what kwargs suupported
print("Supported kwargs for biomes: ", biomes.list_supported_params())
# and then
biomes = biomes.filter(
    page_size=15,
    max_depth=6,
)
print("Filtered url: ", biomes.request_url)

Init url:  https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page_size=50
Supported kwargs for biomes:  ['biome_lineage', 'max_depth', 'page', 'page_size']
Filtered url:  https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page_size=15&max_depth=6


## Option 2. MGnipy

The `MGnipy` client offers a unified interface to access various MGnify API endpoints, including biomes. This approach is convenient if you want to manage multiple types of queries or resources through a single client object.

- Instantiate `MGnipy` to configure your API access and manage requests.
- Use `.biomes` to create a biome query with your desired parameters.
- You can use the same filtering and previewing methods as with the proxy, such as `filter()`, `list_parameters()`, and `explain()`.

This method is ideal for users who prefer an object-oriented workflow and may want to extend their analysis to other MGnify resources beyond biomes.

In [3]:
from mgnipy import MGnipy

# init
mg = MGnipy(
    # configuration
)

# access proxy
biomes = mg.biomes(
    page_size=50,
)
print("Init url: ", biomes.request_url)
# if not sure what kwargs suupported
print("Supported kwargs for biomes: ", biomes.list_supported_params())
# and then
biomes = biomes.filter(
    page_size=15,
    max_depth=6,
)
print("Filtered url: ", biomes.request_url)

Init url:  https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page_size=50
Supported kwargs for biomes:  ['biome_lineage', 'max_depth', 'page', 'page_size']
Filtered url:  https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page_size=15&max_depth=6


## Previewing your requests

There is an optional intermediary step to
- `.preview()` the first page of results,  or
- `.dry_run()` to print the number of pages and records to request
- `.explain()` to print the planned request urls
before `.get()`ting all the result pages.

In [4]:
biomes.explain(head=5)
# or
# biomes.dry_run()
# or
# biomes.preview()

Planning the API call with params:
{'page_size': 15, 'max_depth': 6}
Total pages to retrieve: 33
Total records to retrieve: 492
https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page=1
https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page=2
https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page=3
https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page=4
https://www.ebi.ac.uk/metagenomics/api/v2/biomes/?page=5


## Carry out requests
If happy with the plan, proceed with the async get requests.

In [5]:
# asynchronously get the data
await biomes.aget()

Retrieving pages: 100%|██████████| 33/33 [00:00<00:00, 107.19it/s]


## Exploring the results

Different means to access the retireved results

### as a list

In [6]:
biomes.to_list()[:5]

[{'biome_name': 'Metal', 'lineage': 'root:Engineered:Bioremediation:Metal'},
 {'biome_name': 'Persistent organic pollutants (POP)',
  'lineage': 'root:Engineered:Bioremediation:Persistent organic pollutants (POP)'},
 {'biome_name': 'Polycyclic aromatic hydrocarbons',
  'lineage': 'root:Engineered:Bioremediation:Polycyclic aromatic hydrocarbons'},
 {'biome_name': 'Terephthalate',
  'lineage': 'root:Engineered:Bioremediation:Terephthalate'},
 {'biome_name': 'Wastewater',
  'lineage': 'root:Engineered:Bioremediation:Terephthalate:Wastewater'}]

### as a pandas dataframe

In [7]:
biomes.to_df().head()

,biome_name,lineage
0,Metal,root:Engineered:Bioremediation:Metal
1,Persistent organic pollutants (POP),root:Engineered:Bioremediation:Persistent orga...
2,Polycyclic aromatic hydrocarbons,root:Engineered:Bioremediation:Polycyclic arom...
3,Terephthalate,root:Engineered:Bioremediation:Terephthalate
4,Wastewater,root:Engineered:Bioremediation:Terephthalate:W...


### as a dictionary
where each key is a page

In [8]:
# look at first 5 records of page 1
biomes.results[1][:5]

[{'biome_name': 'root', 'lineage': 'root'},
 {'biome_name': 'Control', 'lineage': 'root:Control'},
 {'biome_name': 'Engineered', 'lineage': 'root:Engineered'},
 {'biome_name': 'Biogas plant', 'lineage': 'root:Engineered:Biogas plant'},
 {'biome_name': 'Wet fermentation',
  'lineage': 'root:Engineered:Biogas plant:Wet fermentation'}]

Specific to the biomes, results can also be visualized as a tree "print" "hshow" or "vshow"

In [9]:
biomes.show_tree()

TypeError: object of type 'NoneType' has no len()

In [ ]:
biomes[1]

TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'

In [ ]:
their_studies.explain(head=5)

Planning the API call with params:
{'biome_lineage': 'root:Engineered:Wastewater:Industrial wastewater'}
Total pages to retrieve: 1
Total records to retrieve: 24
https://www.ebi.ac.uk/metagenomics/api/v2/studies/?biome_lineage=root%3AEngineered%3AWastewater%3AIndustrial+wastewater&page=1


In [ ]:
their_studies.get()

Retrieving pages: 100%|██████████| 1/1 [00:00<00:00,  4.97it/s]


In [ ]:
study_detail = their_studies[0]

Planning the API call with params:
{'accession': 'MGYS00000777'}
Total pages to retrieve: 1
Total records to retrieve: 1
